In [34]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

#dataset na het opschonen van missing values inlezen
df = pd.read_csv('Leesmateriaal_DU.csv')
df = df.reset_index(drop=True)
df = df.convert_dtypes()
df.dtypes

Titel                                string
Auteur                               string
Korte omschrijving                   string
Type materiaal                       string
Niveau                               string
genre                                string
onderwerp                            string
trefwoorden                          string
taal                                 string
aantal_paginas                        Int64
geschatte_leestijd_min                Int64
bron_url                             string
doelgroep                            string
gegevensstatus                       string
leescomplexiteit_score              Float64
interessecluster                     string
uitgever                             string
publicatiejaar                        Int64
beschikbaar_formaat                  string
cataloguscode                        string
leescomplexiteit_score_ontbreekt    boolean
dtype: object

In [35]:
def maak_zoektekst(df):
    kolommen = [
        "Titel", "Korte omschrijving", "Type materiaal", "Niveau", 
        "genre", "onderwerp", "trefwoorden", "taal", "doelgroep", "interessecluster"
    ]

    zoektekst = df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return zoektekst


df["tekst_gecombineerd"] = maak_zoektekst(df)

df["tekst_gecombineerd"].iloc[[0]]

0    Het dagboek van Anne Frank Dagboek van een Joods meisje dat tijdens de Tweede Wereldoorlog onderduikt in Amsterdam. Haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. Boek 3F Fictie WOII WOII; onderduik Nederlands MBO Verhalen
Name: tekst_gecombineerd, dtype: str

In [36]:
import re
import spacy

nlp = spacy.load("nl_core_news_lg")

In [37]:
df["tekst_lowercase"] = (
    df["tekst_gecombineerd"]
    .str.lower()
    .str.strip()
)

df["tekst_lowercase"].iloc[[0]]

0    het dagboek van anne frank dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam. haar persoonlijke observaties geven een indringend beeld van angst, hoop, groei en het dagelijks leven onder vervolging. boek 3f fictie woii woii; onderduik nederlands mbo verhalen
Name: tekst_lowercase, dtype: str

In [38]:
def tekst_opschonen(tekst):
    # Verwijder losse cijfers, maar behoud bijvoorbeeld 2f en 3f
    tekst = re.sub(r"(?<![A-Za-zÀ-ÿ])\d+(?![A-Za-zÀ-ÿ])", " ", tekst)

    # Verwijder leestekens, maar behoud letters, cijfers en spaties
    tekst = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", tekst)

    # Maak meerdere spaties één spatie
    tekst = re.sub(r"\s+", " ", tekst)

    return tekst.strip()

df["tekst_schoon"] = df["tekst_lowercase"].apply(tekst_opschonen)

df["tekst_schoon"].iloc[[0]]

0    het dagboek van anne frank dagboek van een joods meisje dat tijdens de tweede wereldoorlog onderduikt in amsterdam haar persoonlijke observaties geven een indringend beeld van angst hoop groei en het dagelijks leven onder vervolging boek 3f fictie woii woii onderduik nederlands mbo verhalen
Name: tekst_schoon, dtype: str

In [39]:
df["tokens"] = df["tekst_schoon"].apply(
    lambda tekst: [token.text for token in nlp(tekst)]
)

df["tokens"].iloc[[0]]

0    [het, dagboek, van, anne, frank, dagboek, van, een, joods, meisje, dat, tijdens, de, tweede, wereldoorlog, onderduikt, in, amsterdam, haar, persoonlijke, observaties, geven, een, indringend, beeld, van, angst, hoop, groei, en, het, dagelijks, leven, onder, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo, verhalen]
Name: tokens, dtype: object

In [40]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stopwoorden = set(stopwords.words("dutch"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\addov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
def verwijder_stopwoorden(tekst):
    tokens = tekst.split()

    return [
        token
        for token in tokens
        if token not in stopwoorden
    ]

df["tokens_zonder_stopwoorden"] = df["tekst_schoon"].apply(
    verwijder_stopwoorden
)

df["tokens_zonder_stopwoorden"].iloc[[0]]

0    [dagboek, anne, frank, dagboek, joods, meisje, tijdens, tweede, wereldoorlog, onderduikt, amsterdam, persoonlijke, observaties, geven, indringend, beeld, angst, hoop, groei, dagelijks, leven, vervolging, boek, 3f, fictie, woii, woii, onderduik, nederlands, mbo, verhalen]
Name: tokens_zonder_stopwoorden, dtype: object

In [42]:
def lemmatiseer(tekst):
    doc = nlp(tekst)

    resultaat = []

    for token in doc:
        is_niveau = re.fullmatch(r"\d+[a-zA-Z]", token.text)

        if token.text.lower() in stopwoorden:
            continue

        if token.is_alpha:
            resultaat.append(token.lemma_)
        elif is_niveau:
            resultaat.append(token.text.upper())

    return resultaat

df["tokens_basisvorm"] = df["tekst_schoon"].apply(lemmatiseer)

df["tokens_basisvorm"].iloc[[0]]

0    [dagboek, Anne, Frank, dagboek, joods, meisje, tijdens, twee, wereldoorlog, onderduiken, Amsterdam, persoonlijk, observatie, geven, indringen, beeld, angst, hopen, groei, dagelijks, leven, vervolging, boek, 3F, fictie, woii, woii, onderduiken, Nederlands, mbo, verhaal]
Name: tokens_basisvorm, dtype: object

In [43]:
df["tekst_nlp"] = df["tokens_basisvorm"].apply(
    lambda tokens: " ".join(tokens)
)

df["tekst_nlp"].iloc[0:3]

0    dagboek Anne Frank dagboek joods meisje tijdens twee wereldoorlog onderduiken Amsterdam persoonlijk observatie geven indringen beeld angst hopen groei dagelijks leven vervolging boek 3F fictie woii woii onderduiken Nederlands mbo verhaal
1                                           blauw water spannend thriller waarin vrouw confronteren stalking bedreiging terwijl spanning oplopen zetten zichzelf gezin beschermen boek 3F fictie spanning spanning stalking Nederlands mbo verhaal
2                                     boer vinden vrouw toegankelijk verhaal liefde relatie zoektocht passen partner plattelandsleven vormen decor herkenbaar menselijk ervaring keuze boek 2F fictie liefde liefde relatie Nederlands mbo verhaal
Name: tekst_nlp, dtype: str

In [44]:
#Maak code waar je de studenten informatie nu ook nlp maakt 
#vergelijk dan de nlp van de boeken en de nlp van de studenten informatie met elkaar 
#en laat zien welke boeken het meest relevant zijn voor de studenten informatie

In [45]:
#dataset na het opschonen van missing values inlezen
dfStudenten = pd.read_csv('Studenten.csv')
dfStudenten = dfStudenten.convert_dtypes()
dfStudenten.dtypes

student_id                string
leeftijdscategorie        string
leesniveau                string
taalniveau                string
interesses                string
voorkeur_materiaaltype    string
voorkeur_genre            string
max_leestijd_min           Int64
voorkeur_taal             string
dtype: object

In [46]:
def maak_zoektekst(df):
    kolommen = [
        "leesniveau", "interesses", "voorkeur_materiaaltype",
        "voorkeur_genre", "voorkeur_taal",
    ]

    zoektekst = df[kolommen].apply(
        lambda rij: " ".join(
            rij.dropna().astype(str).str.strip()
        ),
        axis=1
    )

    return zoektekst

dfStudenten["tekst_gecombineerd"] = maak_zoektekst(dfStudenten)

dfStudenten["tekst_gecombineerd"].iloc[[0]]

0    2F kunstmatige intelligentie, relaties Boek informatief Nederlands
Name: tekst_gecombineerd, dtype: str

In [47]:
dfStudenten["tekst_lowercase"] = (
    dfStudenten["tekst_gecombineerd"]
    .str.lower()
    .str.strip()
)

dfStudenten["tekst_lowercase"].iloc[[0]]

0    2f kunstmatige intelligentie, relaties boek informatief nederlands
Name: tekst_lowercase, dtype: str

In [48]:
def tekst_opschonen(tekst):
    # Verwijder losse cijfers, maar behoud bijvoorbeeld 2f en 3f
    tekst = re.sub(r"(?<![A-Za-zÀ-ÿ])\d+(?![A-Za-zÀ-ÿ])", " ", tekst)

    # Verwijder leestekens, maar behoud letters, cijfers en spaties
    tekst = re.sub(r"[^A-Za-zÀ-ÿ0-9\s]", " ", tekst)

    # Maak meerdere spaties één spatie
    tekst = re.sub(r"\s+", " ", tekst)

    return tekst.strip()

dfStudenten["tekst_schoon"] = dfStudenten["tekst_lowercase"].apply(tekst_opschonen)

dfStudenten["tekst_schoon"].iloc[[0]]

0    2f kunstmatige intelligentie relaties boek informatief nederlands
Name: tekst_schoon, dtype: str

In [49]:
dfStudenten["tokens"] = dfStudenten["tekst_schoon"].apply(
    lambda tekst: [token.text for token in nlp(tekst)]
)

dfStudenten["tokens"].iloc[[0]]

0    [2f, kunstmatige, intelligentie, relaties, boek, informatief, nederlands]
Name: tokens, dtype: object

In [50]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stopwoorden = set(stopwords.words("dutch"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\addov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [51]:
def verwijder_stopwoorden(tekst):
    tokens = tekst.split()

    return [
        token
        for token in tokens
        if token not in stopwoorden
    ]

dfStudenten["tokens_zonder_stopwoorden"] = dfStudenten["tekst_schoon"].apply(
    verwijder_stopwoorden
)

dfStudenten["tokens_zonder_stopwoorden"].iloc[[0]]

0    [2f, kunstmatige, intelligentie, relaties, boek, informatief, nederlands]
Name: tokens_zonder_stopwoorden, dtype: object

In [52]:
def lemmatiseer(tekst):
    doc = nlp(tekst)

    resultaat = []

    for token in doc:
        is_niveau = re.fullmatch(r"\d+[a-zA-Z]", token.text)

        if token.text.lower() in stopwoorden:
            continue

        if token.is_alpha:
            resultaat.append(token.lemma_)
        elif is_niveau:
            resultaat.append(token.text.upper())

    return resultaat

dfStudenten["tokens_basisvorm"] = dfStudenten["tekst_schoon"].apply(lemmatiseer)

dfStudenten["tokens_basisvorm"].iloc[[0]]

0    [2F, kunstmatig, intelligentie, relatie, boek, informatief, Nederlands]
Name: tokens_basisvorm, dtype: object

In [53]:
dfStudenten["tekst_nlp"] = dfStudenten["tokens_basisvorm"].apply(
    lambda tokens: " ".join(tokens)
)

dfStudenten["tekst_nlp"].iloc[0:3]

0    2F kunstmatig intelligentie relatie boek informatief Nederlands
1     2F reizen persoonlijk ontwikkeling boek informatief Nederlands
2           2F technologie humor sport mengen informatief Nederlands
Name: tekst_nlp, dtype: str

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

# Alleen fitten op de boeken
tfidf_df = vectorizer.fit_transform(df["tekst_nlp"])

# Dezelfde vectorizer gebruiken voor studenten
tfidf_dfStudenten = vectorizer.transform(dfStudenten["tekst_nlp"])

student_book_similarity = cosine_similarity(
    tfidf_dfStudenten,
    tfidf_df
)

student_book_similarity.shape

(100, 277)

In [55]:
student_index = 0
top_n = 5

# Scores van deze student voor alle boeken
scores = student_book_similarity[student_index]

# Sorteer op hoogste cosine similarity
best_indices = scores.argsort()[::-1][:top_n]

recommendations = df.iloc[best_indices].copy()
recommendations["similarity"] = scores[best_indices]

recommendations[["Titel", "similarity"]]

,Titel,similarity
44,Hotel rozenstok,0.314237
12,Een echte man,0.273485
70,De zweetvoetenman,0.269473
2,Boer vindt vrouw,0.239124
73,Wie zit in de gevangenis? En andere bijzondere weetjes over misdaad en straf.,0.205905


In [56]:
all_recommendations = {}

for student_index in range(len(dfStudenten)):
    scores = student_book_similarity[student_index]

    best_indices = scores.argsort()[::-1][:5]

    all_recommendations[student_index] = df.iloc[
        best_indices
    ].copy()

    all_recommendations[student_index]["similarity"] = (
        scores[best_indices]
    )

all_recommendations[0][["Titel", "similarity"]]

,Titel,similarity
44,Hotel rozenstok,0.314237
12,Een echte man,0.273485
70,De zweetvoetenman,0.269473
2,Boer vindt vrouw,0.239124
73,Wie zit in de gevangenis? En andere bijzondere weetjes over misdaad en straf.,0.205905


In [57]:
def normaliseer_tekst(waarde):
    if pd.isna(waarde):
        return ""

    tekst = str(waarde).lower().strip()
    return tekst

In [58]:
def is_relevant(student, boek):
    interesse_match = False
    genre_match = False
    taal_match = False

    # 1. Interesse tegen onderwerp/trefwoorden/omschrijving
    student_interesses = [
        normaliseer_tekst(interesse)
        for interesse in str(
            student["interesses"]
        ).split(",")
        if normaliseer_tekst(interesse)
    ]

    boek_tekst = " ".join([
        normaliseer_tekst(boek["onderwerp"]),
        normaliseer_tekst(boek["trefwoorden"]),
        normaliseer_tekst(boek["Korte omschrijving"])
    ])

    interesse_match = any(
        interesse in boek_tekst
        for interesse in student_interesses
    )

    # 2. Genre
    student_genre = normaliseer_tekst(
        student["voorkeur_genre"]
    )
    boek_genre = normaliseer_tekst(boek["genre"])

    genre_match = (
        student_genre == ""
        or student_genre == "geen voorkeur"
        or student_genre in boek_genre
        or pd.isna(boek["genre"])
    )

    # 3. Taal
    student_taal = normaliseer_tekst(
        student["voorkeur_taal"]
    )
    boek_taal = normaliseer_tekst(boek["taal"])

    taal_match = (
        student_taal == ""
        or not student_taal
        or student_taal == boek_taal
        or pd.isna(boek["taal"])
    )

    return (
    interesse_match
    or (
        genre_match
        and taal_match
    )
)

In [59]:
relevant_codes_per_student = {}

for student_index, student in dfStudenten.iterrows():
    relevante_codes = set()

    for _, boek in df.iterrows():
        if is_relevant(student, boek):
            relevante_codes.add(
                boek["cataloguscode"]
            )

    relevant_codes_per_student[student_index] = relevante_codes

In [60]:
import math

dfInteracties = pd.read_csv("Interacties.csv")

precision_rijen = []
ndcg_rijen = []

for student_index, student in dfStudenten.iterrows():
    scores = student_book_similarity[student_index]
    relevante_codes = relevant_codes_per_student[student_index]

    seen_codes = set(
        dfInteracties.loc[
            dfInteracties["student_id"] == student["student_id"],
            "cataloguscode"
        ]
    )

    beschikbare_indices = [
        index
        for index, code in enumerate(df["cataloguscode"])
        if code not in seen_codes
    ]

    top_indices = sorted(
        beschikbare_indices,
        key=lambda index: scores[index],
        reverse=True
    )[:5]

    # Precision@5
    recommended_codes = set(
        df.iloc[top_indices]["cataloguscode"]
    )
    aantal_hits = len(recommended_codes & relevante_codes)

    precision_rijen.append({
        "student_id": student["student_id"],
        "aantal_hits": aantal_hits,
        "precision_at_5": aantal_hits / 5,
        "relevant_books": len(relevante_codes)
    })

    # NDCG@5: behoud de volgorde van de aanbevelingen
    top_codes = df.iloc[top_indices]["cataloguscode"].tolist()
    relevantie_top_5 = [
        int(code in relevante_codes)
        for code in top_codes
    ]

    dcg = sum(
        (2**relevantie - 1) / math.log2(positie + 2)
        for positie, relevantie in enumerate(relevantie_top_5)
    )

    beschikbare_codes = {
        df.iloc[index]["cataloguscode"]
        for index in beschikbare_indices
    }
    aantal_relevant_beschikbaar = len(
        relevante_codes & beschikbare_codes
    )
    ideale_lengte = min(
        aantal_relevant_beschikbaar,
        len(top_indices)
    )

    idcg = sum(
        1 / math.log2(positie + 2)
        for positie in range(ideale_lengte)
    )

    ndcg_at_5 = dcg / idcg if idcg > 0 else float("nan")

    ndcg_rijen.append({
        "student_id": student["student_id"],
        "ndcg_at_5": ndcg_at_5
    })

In [61]:
precision_tabel = pd.DataFrame(precision_rijen)
ndcg_tabel = pd.DataFrame(ndcg_rijen)

gemiddelde_precision_at_5 = precision_tabel["precision_at_5"].mean()
gemiddelde_ndcg_at_5 = ndcg_tabel["ndcg_at_5"].mean()

print("Gemiddelde Precision@5:", round(gemiddelde_precision_at_5, 4))
print("Gemiddelde NDCG@5:", round(gemiddelde_ndcg_at_5, 4))
print(
    "Studenten meegenomen in NDCG@5:",
    ndcg_tabel["ndcg_at_5"].notna().sum()
)

Gemiddelde Precision@5: 0.934
Gemiddelde NDCG@5: 0.938
Studenten meegenomen in NDCG@5: 100
